## Подключение к исследованию


Последовательно в терминал вводим следующие команды 
cd "C:\Users\truni\OneDrive\Рабочий стол\LifeStyle\_Иннополис\_1 - Attestation"


In [ ]:
# Проверяем совместимость версий и используемые библиотеки 

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import requests # Python-библиотека для HTTP-запросов. Она позволяет из кода обращаться к веб-сайтам и API
import duckdb # позволяет создавать базы данных, хранить и анализировать большие объёмы информации с помощью SQL непосредственно из Python.
import sys # позволяет Python-программе получать информацию о среде, в которой она работает, и взаимодействовать с самим интерпретатором Python
import openpyxl # библиотека по выгрузке данных в excel для ручной проверки 



print("Версии библиотек Python (проверка совместимости виртуальной среды):")
print(f"numpy:      {np.__version__}")
print(f"pandas:     {pd.__version__}")
print(f"matplotlib: {matplotlib.__version__}")
print(f"seaborn: {sns.__version__}")
print(f"requests: {requests.__version__}")
print("Версия DuckDB:", duckdb.__version__)
print(f"Версия sys:", {sys.version})
print(openpyxl.__version__)

# Проблема с загрузкой scipy - большой пинг и обрыв (надо разобраться отдельно)

In [ ]:
# Как теперь подключаться в Jupyter Notebook? 
# В начале любого ноутбука, который запускается из корневой папки проекта, достаточно написать:
from project_config import (
    connect_db,
    load_csv,
    list_csv,
    list_tables
)

# Подключаемся к существующей DuckDB
con = connect_db(read_only=True)

# Проверяем таблицы
display(list_tables(con))

# Получаем список исходных CSV
print("Исходные CSV:", list_csv("raw"))

# Получаем список агрегированных CSV
print("Обработанные CSV:", list_csv("processed"))

# Загружаем исходные данные
games = load_csv("games-index.csv")

# Загружаем готовый рейтинг игр
ranking = load_csv(
    "steam_games_ranking.csv",
    folder="processed"
)

print("Исходные игры:", games.shape)
print("Рейтинг игр:", ranking.shape)


In [ ]:
# Теперь для получения любого агрегата понадобится всего одна строка:
genre_lift = load_csv(
    "steam_genre_lift_filtered.csv",
    folder="processed"
)

# А для выполнения SQL-запроса:
df = con.execute("""
    SELECT *
    FROM steam_app_details
    LIMIT 10
""").df()

display(df)

# Если новое подключение выдаст ранее встречавшуюся ошибку different configuration, используй уже открытое con 
# либо закрой остальные соединения. Не нужно создавать новое подключение в каждой ячейке ноутбука.

In [ ]:

from project_config import (
    DB_PATH,
    list_csv,
    list_tables
)

print("=== ПРОВЕРКА ПРОЕКТА ===")

print("База существует:", DB_PATH.is_file())

raw_files = list_csv("raw")
processed_files = list_csv("processed")

print("Исходных CSV:", len(raw_files))
print("Обработанных CSV:", len(processed_files))

# Используем уже открытое соединение con
objects = list_tables(con)

print("Объектов DuckDB:", len(objects))
display(objects)

games_count = con.execute("""
    SELECT COUNT(DISTINCT app_id)
    FROM steam_app_details
""").fetchone()[0]

print("Игр в основной выборке:", games_count)

if games_count == 2000:
    print("Основная выборка соответствует ожидаемой.")
else:
    print("Внимание: размер выборки изменился.")


In [ ]:
# Заранее скаченны .csv файлы с данным с портала Steam 
from pathlib import Path                        # инструмент Python для работы с путями и файлами
import pandas as pd

# Папка с исходными данными
data_dir = Path("data/raw")                     # указывает, где находятся исходные данные (сырые данные)

# Находим все CSV-файлы
csv_files = sorted(data_dir.glob("*.csv"))      # сортирует найденные файлы по имени

print(f"Найдено CSV-файлов: {len(csv_files)}")  #перебирает файлы по одному.
print("Если все 4 файла выведены, то все ОК - самопроверка")

for file in csv_files:
    print(file.name)

In [ ]:
# Создаем для каждого файла отдельный DataFrame загружая ранее скаченные .csv-файлы 
# ссылка на ресурс: https://thegamecensus.com/data?utm_source=chatgpt.com

games = pd.read_csv(data_dir / "games-index.csv")

ccu = pd.read_csv(data_dir / "ccu-daily-top2000.csv")

health = pd.read_csv(data_dir / "health-tiers.csv")

sales = pd.read_csv(data_dir / "sales-calibration.csv")

# Собираем данные в единный Датасет в виде словаря 
datasets = {
    "Игры": games,
    "Активность": ccu,
    "Состояние": health,
    "Продажи": sales
}
# Выводим параметры по числу столбцов и строк 
for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} строк, {df.shape[1]} столбцов")

In [ ]:

from pathlib import Path
import duckdb
import pandas as pd

# Корневая папка проекта
PROJECT_DIR = Path(
    r"C:\Users\truni\OneDrive\Рабочий стол\LifeStyle\_Иннополис\_1 - Attestation\data\database"
)

# Путь к базе данных
DB_PATH = PROJECT_DIR / "steam_analytics.duckdb"

# Проверяем существование файла
print("Путь к базе:", DB_PATH)
print("База существует:", DB_PATH.is_file())

# Подключаемся
assert DB_PATH.is_file(), f"База не найдена: {DB_PATH}"

con = duckdb.connect(
    database=str(DB_PATH),
    read_only=True
)

print("Подключение успешно!")
print("Версия DuckDB:", duckdb.__version__)
